# Image Captioning - Kaggle Training Run

**ResNet50 frozen encoder + LSTM decoder on COCO 2017.**

Clones the repo, mounts data, trains with auto-resume, pushes best checkpoint
to HF Hub, runs full pycocoevalcap on val2017.

---
**PRE-REQUISITES (one-time):**

1. Kaggle Dataset `coco-features` with cached encoder features + `vocab.json`
   (created by `notebooks/cache_features_kaggle.ipynb`).
2. COCO 2017 Dataset (images + annotations).
3. HF Hub token as Kaggle Secret (`HF_TOKEN`).
---

## 1. Setup - repo, env, and data mounts

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/MohamedShakshak/Image-Captioning.git"
CONFIG_PATH = "configs/default.yaml"

def _inside_repo() -> bool:
    try:
        out = subprocess.run(
            ["git", "rev-parse", "--show-toplevel"],
            capture_output=True, text=True, timeout=5,
        )
        return out.returncode == 0 and out.stdout.strip().endswith("Image-Captioning")
    except Exception:
        return False

if _inside_repo():
    print("Already inside repo - skip clone.")
elif os.path.isdir("Image-Captioning"):
    print("Repo dir exists - cd into it.")
    %cd Image-Captioning
else:
    !git clone --depth 1 {REPO_URL}
    %cd Image-Captioning

!pip install -e . -q
!pip install -e .[app] -q

_src = os.path.join(os.getcwd(), "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

import torch; print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

In [ ]:
# Add these Datasets via Kaggle UI: coco-2017, coco-features

COCO_ROOT = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"
ANNO_TRAIN = f"{COCO_ROOT}/annotations/captions_train2017.json"
ANNO_VAL = f"{COCO_ROOT}/annotations/captions_val2017.json"
FEATURES_DIR = "/kaggle/input/datasets/mohamedshakshak/resnet50"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
VOCAB_PATH = f"{FEATURES_DIR}/vocab.json"

for p in [COCO_ROOT, ANNO_TRAIN, ANNO_VAL, FEATURES_DIR]:
    assert os.path.exists(p), f"Missing mount: {p}"
    print(f"OK: {p}")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 2. Training

In [ ]:
import sys
sys.argv = [
    "train.py",
    "--config", CONFIG_PATH,
    f"data.coco-root={COCO_ROOT}",
    f"data.annotation-train={ANNO_TRAIN}",
    f"data.annotation-val={ANNO_VAL}",
    f"data.features-dir={FEATURES_DIR}",
    f"data.vocab-path={VOCAB_PATH}",
    f"checkpoint.dir={CHECKPOINT_DIR}",
    "checkpoint.save-latest=true",
    "checkpoint.save-best=true",
    "train.amp=false",
]

from train import train as run_training
from config import parse_args

cfg, _ = parse_args()
print("Config:", cfg)
run_training(cfg)

## 3. Full Evaluation on COCO val2017

All 5 metrics (BLEU-1..4, CIDEr, ROUGE-L, METEOR) via pycocoevalcap.
CIDEr/METEOR require Java (pre-installed on Kaggle).

In [ ]:
sys.argv = [
    "evaluate.py",
    "--config", CONFIG_PATH,
    f"data.coco-root={COCO_ROOT}",
    f"data.annotation-train={ANNO_TRAIN}",
    f"data.annotation-val={ANNO_VAL}",
    f"data.features-dir={FEATURES_DIR}",
    f"data.vocab-path={VOCAB_PATH}",
    f"checkpoint.dir={CHECKPOINT_DIR}",
    "eval.full-metrics=true",
    "eval.beam-size=3",
]

from evaluate import evaluate as run_eval

cfg, _ = parse_args()
run_eval(cfg)

## 4. Results & Upload

In [ ]:
!python scripts/plot_curves.py --metrics metrics.json --out /kaggle/working/curves.png
import shutil
for f in ["metrics.json", "eval_out/preds.json", "eval_out/gts.json", "curves.png"]:
    if os.path.exists(f):
        shutil.copy(f, "/kaggle/working/")
        print(f"Saved /kaggle/working/{f}")

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN")
HF_REPO = "MohamedShakshak/image-captioning-pytorch"

if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_REPO, repo_type="model", exist_ok=True)

    best_path = f"{CHECKPOINT_DIR}/best.pt"
    if os.path.exists(best_path):
        api.upload_file(path_or_fileobj=best_path, path_in_repo="best.pt",
                        repo_id=HF_REPO, repo_type="model")
        print(f"Uploaded best.pt to {HF_REPO}")

    if os.path.exists(VOCAB_PATH):
        api.upload_file(path_or_fileobj=VOCAB_PATH, path_in_repo="vocab.json",
                        repo_id=HF_REPO, repo_type="model")
else:
    print("No HF_TOKEN - results remain in /kaggle/working/.")

## 5. Summary

- `/kaggle/working/checkpoints/best.pt` - best model
- `/kaggle/working/checkpoints/latest.pt` - latest (for resume)
- `/kaggle/working/metrics.json` - per-epoch log
- `/kaggle/working/curves.png` - training curve
- `MohamedShakshak/image-captioning-pytorch` - uploaded on HF Hub

In [ ]:
print("Done.")